=================================================
Milestone 3

Nama  : Christoper Leonardo Yosseri
Batch : CODA-015-RMT

Program ini dirancang untuk mendeteksi anomali data secara dini, melakukan pembersihan data dari duplikat, dan menghasilkan dokumentasi data (Data Docs) secara otomatis
=================================================

# 1. Import Library

In [ ]:
!pip install kagglehub pandas
!pip install -q "great-expectations==0.18.19"

import kagglehub
import pandas as pd
import great_expectations as gx
from great_expectations.data_context import FileDataContext
import os

# 2. Load Data

In [2]:
# 1. Download dan Load Data langsung ke Pandas
df = kagglehub.load_dataset(
    kagglehub.KaggleDatasetAdapter.PANDAS,
    "junaid512/bmw-car-sales-classification-dataset",
    "BMW_Car_Sales_Classification.csv"
)

df.head()

,Model,Year,Region,Color,Fuel_Type,Transmission,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume,Sales_Classification
0,5 Series,2016,Asia,Red,Petrol,Manual,3.5,151748,98740,8300,High
1,i8,2013,North America,Red,Hybrid,Automatic,1.6,121671,79219,3428,Low
2,5 Series,2022,North America,Blue,Petrol,Automatic,4.5,10991,113265,6994,Low
3,X3,2024,Middle East,Blue,Petrol,Automatic,1.7,27255,60971,4047,Low
4,7 Series,2020,South America,Black,Diesel,Manual,2.1,122131,49898,3080,Low


# 3. Eksplorasi Data

In [3]:
# Cek banyak data dan total kolom
df.shape

(50000, 11)

In [4]:
# Cek info data
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Model                 50000 non-null  object 
 1   Year                  50000 non-null  int64  
 2   Region                50000 non-null  object 
 3   Color                 50000 non-null  object 
 4   Fuel_Type             50000 non-null  object 
 5   Transmission          50000 non-null  object 
 6   Engine_Size_L         50000 non-null  float64
 7   Mileage_KM            50000 non-null  int64  
 8   Price_USD             50000 non-null  int64  
 9   Sales_Volume          50000 non-null  int64  
 10  Sales_Classification  50000 non-null  object 
dtypes: float64(1), int64(4), object(6)
memory usage: 4.2+ MB


In [5]:
df.describe()

,Year,Engine_Size_L,Mileage_KM,Price_USD,Sales_Volume
count,50000.000000,50000.000000,50000.000000,50000.000000,50000.000000
mean,2017.015700,3.247180,100307.203140,75034.600900,5067.514680
std,4.324459,1.009078,57941.509344,25998.248882,2856.767125
min,2010.000000,1.500000,3.000000,30000.000000,100.000000
25%,2013.000000,2.400000,50178.000000,52434.750000,2588.000000
50%,2017.000000,3.200000,100388.500000,75011.500000,5087.000000
75%,2021.000000,4.100000,150630.250000,97628.250000,7537.250000
max,2024.000000,5.000000,199996.000000,119998.000000,9999.000000


In [6]:
# Cek data yang nilainya null
jumlah_data_null = df.isnull().sum()
jumlah_percent = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
            'Total Data Hilang' : jumlah_data_null,
            'Percent' : jumlah_percent
})
print(missing_df)

                      Total Data Hilang  Percent
Model                                 0      0.0
Year                                  0      0.0
Region                                0      0.0
Color                                 0      0.0
Fuel_Type                             0      0.0
Transmission                          0      0.0
Engine_Size_L                         0      0.0
Mileage_KM                            0      0.0
Price_USD                             0      0.0
Sales_Volume                          0      0.0
Sales_Classification                  0      0.0


In [7]:
# Cek data duplikat
jumlah_duplikat = df.duplicated().sum()
print(f'Jumlah baris duplikat: {jumlah_duplikat}')

Jumlah baris duplikat: 0


# 4 Transformasi Data 

Ubah Nama Kolom

In [8]:
# mengubah huruf kapital menjadi kecil
df.columns = [col.lower() for col in df.columns]
df.head(5)

,model,year,region,color,fuel_type,transmission,engine_size_l,mileage_km,price_usd,sales_volume,sales_classification
0,5 Series,2016,Asia,Red,Petrol,Manual,3.5,151748,98740,8300,High
1,i8,2013,North America,Red,Hybrid,Automatic,1.6,121671,79219,3428,Low
2,5 Series,2022,North America,Blue,Petrol,Automatic,4.5,10991,113265,6994,Low
3,X3,2024,Middle East,Blue,Petrol,Automatic,1.7,27255,60971,4047,Low
4,7 Series,2020,South America,Black,Diesel,Manual,2.1,122131,49898,3080,Low


Hapus duplikat

In [9]:
# Buang data duplikat
# .reset_index(drop=True) merapihkan angka yang muncul di index lama agar tau berapa total data bersih 
df_clean = df.drop_duplicates().reset_index(drop=True)
df_clean

,model,year,region,color,fuel_type,transmission,engine_size_l,mileage_km,price_usd,sales_volume,sales_classification
0,5 Series,2016,Asia,Red,Petrol,Manual,3.5,151748,98740,8300,High
1,i8,2013,North America,Red,Hybrid,Automatic,1.6,121671,79219,3428,Low
2,5 Series,2022,North America,Blue,Petrol,Automatic,4.5,10991,113265,6994,Low
3,X3,2024,Middle East,Blue,Petrol,Automatic,1.7,27255,60971,4047,Low
4,7 Series,2020,South America,Black,Diesel,Manual,2.1,122131,49898,3080,Low
...,...,...,...,...,...,...,...,...,...,...,...
49995,i3,2014,Asia,Red,Hybrid,Manual,4.6,151030,42932,8182,High
49996,i3,2023,Middle East,Silver,Electric,Manual,4.2,147396,48714,9816,High
49997,5 Series,2010,Middle East,Red,Petrol,Automatic,4.5,174939,46126,8280,High
49998,i3,2020,Asia,White,Electric,Automatic,3.8,3379,58566,9486,High


# 4.3 Simpan Data Bersih

In [10]:
# Simpan df_clean ke file CSV fisik
# df_clean.to_csv('P2M3_christoper_leonardo_yosseri_data_clean.csv', index=False)

In [ ]:
df_clean

,model,year,region,color,fuel_type,transmission,engine_size_l,mileage_km,price_usd,sales_volume,sales_classification
0,5 Series,2016,Asia,Red,Petrol,Manual,3.5,151748,98740,8300,High
1,i8,2013,North America,Red,Hybrid,Automatic,1.6,121671,79219,3428,Low
2,5 Series,2022,North America,Blue,Petrol,Automatic,4.5,10991,113265,6994,Low
3,X3,2024,Middle East,Blue,Petrol,Automatic,1.7,27255,60971,4047,Low
4,7 Series,2020,South America,Black,Diesel,Manual,2.1,122131,49898,3080,Low
...,...,...,...,...,...,...,...,...,...,...,...
49995,i3,2014,Asia,Red,Hybrid,Manual,4.6,151030,42932,8182,High
49996,i3,2023,Middle East,Silver,Electric,Manual,4.2,147396,48714,9816,High
49997,5 Series,2010,Middle East,Red,Petrol,Automatic,4.5,174939,46126,8280,High
49998,i3,2020,Asia,White,Electric,Automatic,3.8,3379,58566,9486,High


# 5 Great Expactation

7 Expectations:
- to be unique
- to be between min_value and max_value
- to be in set
- to be in type list
- to not be null
- table column count to equal

In [12]:
import great_expectations as gx
print(gx.__version__)

0.18.19


### 5.1 Menghubungkan ke Datasource

In [13]:
import great_expectations as gx

# Buat Data Context
context = gx.get_context()

# Tambahkan datasource dari Pandas
datasource_name = 'bmw-data'
datasource = context.sources.add_pandas(datasource_name)

# Tambahkan DataFrame sebagai asset
asset = datasource.add_dataframe_asset(
    name='bmw-car-sales',
    dataframe=df_clean
)

# build batch request
batch_request = asset.build_batch_request()

### 5.2 Buat Expectation Suite

In [14]:
# Buat Expectation Suite
expectation_suite_name = 'bmw-expectation-suite'
context.add_or_update_expectation_suite(expectation_suite_name)

# Validator
validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=expectation_suite_name
)

# Cek validator
validator.head()

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

,model,year,region,color,fuel_type,transmission,engine_size_l,mileage_km,price_usd,sales_volume,sales_classification
0,5 Series,2016,Asia,Red,Petrol,Manual,3.5,151748,98740,8300,High
1,i8,2013,North America,Red,Hybrid,Automatic,1.6,121671,79219,3428,Low
2,5 Series,2022,North America,Blue,Petrol,Automatic,4.5,10991,113265,6994,Low
3,X3,2024,Middle East,Blue,Petrol,Automatic,1.7,27255,60971,4047,Low
4,7 Series,2020,South America,Black,Diesel,Manual,2.1,122131,49898,3080,Low


### 5.3 Expectations

In [29]:
# Expectations 1 
validator.expect_column_values_to_not_be_null('model')

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 50000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
# Expectation 2 - Harga BMW berada di range $30.000 - $120.000
validator.expect_column_values_to_be_between(
    column='price_usd',
    min_value=30000,
    max_value=120000
)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 50000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [17]:
# Expectation 3 - Jenis bahan bakar harus 4 jenis
validator.expect_column_values_to_be_in_set(
    column='fuel_type',
    value_set=['Petrol', 'Hybrid', 'Diesel', 'Electric']
)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 50000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [18]:
# Expectation 4 - kolom Price_USD harus bertipe int64 atau float
validator.expect_column_values_to_be_in_type_list(
    'price_usd', ['int64', 'float'])

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": "int64"
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [19]:
# Expectation 5 - Tidak ada data yang null
validator.expect_column_values_to_not_be_null('model')

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 50000,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [20]:
# Expectation 6 - Apakah rata-rata harga BMW sesuai ekspektasi pasar?
validator.expect_column_mean_to_be_between(
    column='price_usd',
    min_value=70000,
    max_value=80000
)

Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": 75034.6009
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [ ]:
# Expectation 7 - Kolom untuk menentukan depresiasi harga 
validator.expect_column_to_exist(column='year')

Calculating Metrics:   0%|          | 0/2 [00:00<?, ?it/s]

{
  "success": true,
  "result": {},
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

### 5.4 Data Docs

In [22]:
# Build data docs

context.build_data_docs()

{'local_site': 'file://C:\\Users\\Yosseri\\AppData\\Local\\Temp\\tmp417__wwe\\index.html'}